# Cost — HIT-ASR on raw audio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/hit-asr/blob/main/notebooks/timing.ipynb)

The cost measurement of the paper, on each corpus's trio and on raw audio, one clip at a time on one GPU. Three
systems run end to end on the same clips:

| System | What it computes per clip |
|---|---|
| **Best single expert** | its encoder and its decoder |
| **HIT-ASR** | every expert's encoder, the router, and the decoder of the expert it picks, which reuses that expert's encoder output |
| **Decode all + ROVER** | every expert's encoder and decoder, then the vote: what every transcript-fusion method and the oracle need |

Each expert's pipeline is also split into its encoder and its decoder, which is where the saving comes from: HIT-ASR
runs one decoder instead of all of them.

The router is one fold of Table 3, trained on that fold's training half with the selected configuration; the timed
clips are drawn from the half it was tested on. Its choices on raw audio are checked against the choices Table 3
recorded for the same clips, and the encoder states computed from the audio against the stored ones.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, from the records on the Hub | a CPU, seconds |
| **2** | the measurement | a GPU with room for the experts, ~15 min per corpus |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("hitasr") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "hit-asr[paper] @ git+https://github.com/huseyin-karaca/hit-asr.git@main"], check=True)

In [2]:
LEVEL = 1          # 1 or 2
DEVICE = "cuda"    # level 2 measures on it

if LEVEL >= 2:     # the experts' own stack
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets<4.0", "jiwer", "librosa", "soundfile",
                    "transformers>=4.57", "accelerate", "sentencepiece", "torchaudio", "mistral-common[audio]", "peft"],
                   check=True)

from hitasr.configs import TIMING
from hitasr.timing import Timing
from labkit.pretty import show

timing = Timing(TIMING, level=LEVEL, device=DEVICE)

## 1. Per expert

Milliseconds per clip at batch 1. `decoder_share` is the part of an expert's pipeline HIT-ASR avoids for every expert
it does not pick; `same_text` is the share of clips on which decoding from the reused encoder output gave the
pipeline's own transcript; `label_gpu_h` estimates the one-off labelling pass over the whole corpus; `frame_cos` is
the median cosine between the encoder states computed from the audio and the stored ones.

In [3]:
show(*timing.experts())

expert,encode_ms,decode_ms,pipeline_ms,rtf,same_text,reuses_encoder,peak_gb,audio_s_per_clip,encode_ms_median,decode_ms_median,pipeline_ms_median,decoder_share,rtf_batched,label_gpu_h,wer,weights_gb,frame_cos
Cohere-Transcribe,25.9156,21.8125,42.2386,0.0204,1,True,0.0384,2.0663,24.7540,14.8805,35.6083,0.4570,0.0027,0.0789,0.2269,4.1315,0.9999
Granite-Speech-4.1-2b,16.6773,86.9841,96.9141,0.0469,1,True,0.0435,2.0663,17.0197,47.1838,56.7635,0.8391,0.0077,0.2288,0.2019,4.6267,0.9990
Parakeet-TDT-0.6b-v3,11.1043,17.0315,37.4037,0.0181,1,True,0.0443,2.0663,10.6945,12.6862,32.7477,0.6053,0.0019,0.0578,0.3490,1.2541,0.9999


expert,encode_ms,decode_ms,pipeline_ms,rtf,same_text,reuses_encoder,peak_gb,audio_s_per_clip,encode_ms_median,decode_ms_median,pipeline_ms_median,decoder_share,rtf_batched,label_gpu_h,wer,weights_gb,frame_cos
Cohere-Transcribe,27.5318,46.1629,67.4845,0.0096,1,True,0.0964,7.0023,27.0601,41.5514,61.9738,0.6264,0.0016,0.0410,0.1284,4.1315,0.9999
Kyutai-STT-2.6b,15.4706,1963.6901,1965.2663,0.2807,1,False,0.1793,7.0023,15.1366,1879.2033,1895.5589,0.9922,0.0794,2.0563,0.1343,5.6189,0.5775
Parakeet-TDT-0.6b-v3,12.6362,36.4817,57.5069,0.0082,1,True,0.0971,7.0023,12.7090,33.3860,53.5923,0.7427,0.0012,0.0320,0.1361,1.2541,0.9999


expert,encode_ms,decode_ms,pipeline_ms,rtf,same_text,reuses_encoder,peak_gb,audio_s_per_clip,encode_ms_median,decode_ms_median,pipeline_ms_median,decoder_share,rtf_batched,label_gpu_h,wer,weights_gb,frame_cos
Granite-Speech-4.1-2b,17.9664,204.7874,215.1115,0.0348,1,True,0.0625,6.1732,17.8766,160.3875,170.8094,0.9193,0.0058,0.1643,0.2089,4.6267,0.9995
Parakeet-CTC-1.1b,25.2496,0.3333,22.4997,0.0036,1,True,0.0454,6.1732,26.7106,0.3266,22.3788,0.0130,0.0006,0.0158,0.2084,2.1253,1.0000
Parakeet-TDT-0.6b-v3,13.0089,36.0624,57.1681,0.0093,1,True,0.0552,6.1732,13.3967,29.1951,49.9599,0.7349,0.0012,0.0351,0.2029,1.2541,0.9999


expert,encode_ms,decode_ms,pipeline_ms,rtf,same_text,reuses_encoder,peak_gb,audio_s_per_clip,encode_ms_median,decode_ms_median,pipeline_ms_median,decoder_share,rtf_batched,label_gpu_h,wer,weights_gb,frame_cos
Cohere-Transcribe,30.4586,61.8816,82.3154,0.0065,1,True,0.9983,12.6836,29.0359,55.0906,75.1592,0.6701,0.0013,0.0345,0.2415,4.1315,0.9999
Qwen3-ASR-1.7B,12.0180,263.7956,271.4754,0.0214,1,True,2.0393,12.6836,10.9010,224.2236,230.9892,0.9564,0.0068,0.1864,0.3527,4.0762,0.9998
Voxtral-Mini-3B,24.6729,281.1646,302.0002,0.0238,1,True,0.7306,12.6836,22.2608,240.0267,258.2344,0.9193,0.0093,0.2556,0.3463,9.3564,0.9998


## 2. The three systems

`x_best_single` is a system's time over the best single expert's; `saved_vs_decode_all` is the share of the decode-all
time it does not spend. `agreement` is the share of clips on which the router, reading encoder states computed from
the audio, chose what Table 3's fold chose from the stored ones.

In [4]:
show(*timing.systems())

system,kind,encode_ms,decode_ms,total_ms,rtf,wer,route_ms,fuse_ms,x_best_single,saved_vs_decode_all
Cohere-Transcribe (best single),single,25.9156,21.8125,42.2386,0.0204,0.2269,—,—,1.0000,0.7625
Granite-Speech-4.1-2b,single,16.6773,86.9841,96.9141,0.0469,0.2019,—,—,2.2944,0.4551
Parakeet-TDT-0.6b-v3,single,11.1043,17.0315,37.4037,0.0181,0.3490,—,—,0.8855,0.7897
HIT-ASR,hit_asr,51.3456,47.4098,106.4128,0.0515,0.2316,7.6574,—,2.5193,0.4016
Decode all + ROVER,decode_all,—,177.7133,177.8408,0.0861,0.2379,—,0.1274,4.2104,0.0000


system,kind,encode_ms,decode_ms,total_ms,rtf,wer,route_ms,fuse_ms,x_best_single,saved_vs_decode_all
Cohere-Transcribe,single,27.5318,46.1629,67.4845,0.0096,0.1284,—,—,0.0343,0.9680
Kyutai-STT-2.6b (best single),single,15.4706,1963.6901,1965.2663,0.2807,0.1343,—,—,1.0000,0.0670
Parakeet-TDT-0.6b-v3,single,12.6362,36.4817,57.5069,0.0082,0.1361,—,—,0.0293,0.9727
HIT-ASR,hit_asr,49.7826,160.9579,216.9596,0.0310,0.1189,6.2191,—,0.1104,0.8970
Decode all + ROVER,decode_all,—,2105.8933,2106.2919,0.3008,0.1237,—,0.3985,1.0718,0.0000


system,kind,encode_ms,decode_ms,total_ms,rtf,wer,route_ms,fuse_ms,x_best_single,saved_vs_decode_all
Granite-Speech-4.1-2b (best single),single,17.9664,204.7874,215.1115,0.0348,0.2089,—,—,1.0000,0.2804
Parakeet-CTC-1.1b,single,25.2496,0.3333,22.4997,0.0036,0.2084,—,—,0.1046,0.9247
Parakeet-TDT-0.6b-v3,single,13.0089,36.0624,57.1681,0.0093,0.2029,—,—,0.2658,0.8088
HIT-ASR,hit_asr,51.1162,99.7722,156.5267,0.0254,0.1677,5.6383,—,0.7277,0.4764
Decode all + ROVER,decode_all,—,298.4991,298.9395,0.0484,0.1953,—,0.4404,1.3897,0.0000


system,kind,encode_ms,decode_ms,total_ms,rtf,wer,route_ms,fuse_ms,x_best_single,saved_vs_decode_all
Cohere-Transcribe,single,30.4586,61.8816,82.3154,0.0065,0.2415,—,—,0.3032,0.8751
Qwen3-ASR-1.7B (best single),single,12.0180,263.7956,271.4754,0.0214,0.3527,—,—,1.0000,0.5879
Voxtral-Mini-3B,single,24.6729,281.1646,302.0002,0.0238,0.3463,—,—,1.1124,0.5416
HIT-ASR,hit_asr,65.4990,59.3189,132.9753,0.0105,0.2415,8.1574,—,0.4898,0.7982
Decode all + ROVER,decode_all,—,658.1888,658.7878,0.0519,0.3388,—,0.599,2.4267,0.0000


## 3. Across corpora

In [5]:
timing.summary()

corpus,clips,encoders_ms,router_ms,decoder_ms,hit_ms,best_single_ms,decode_all_ms,hit_rtf,best_rtf,all_rtf,x_best_single,saved_vs_all,decoder_share_min,decoder_share_max,router_gflops,router_M,agreement,hit_wer,best_wer,weights_gb_hit,weights_gb_best,label_gpu_h
ami_sdm,100,51.346,7.657,47.410,106.413,42.239,177.841,0.051,0.020,0.086,2.519,0.402,0.457,0.839,0.716,24.479,0.70,0.232,0.227,10.012,4.132,0.365
earnings22,100,49.783,6.219,160.958,216.960,1965.266,2106.292,0.031,0.281,0.301,0.110,0.897,0.626,0.992,1.053,11.857,1.00,0.119,0.134,11.005,5.619,2.129
peoples_speech,100,51.116,5.638,99.772,156.527,215.111,298.940,0.025,0.035,0.048,0.728,0.476,0.013,0.919,3.010,2.029,0.91,0.168,0.209,8.006,4.627,0.215
afrispeech,100,65.499,8.157,59.319,132.975,271.475,658.788,0.010,0.021,0.052,0.490,0.798,0.670,0.956,5.823,24.676,0.49,0.241,0.353,17.564,4.076,0.477


## 4. The router against the number of experts

Latency, parameters and GFLOPs of one fit of the selected router on synthetic inputs of the experts' mean width and
length, for K experts.

In [6]:
timing.scaling()

The router alone against K
  [peoples_speech]
 K  router_ms  params_M  gflops   T  d_in
 3     1.7130    0.6762  0.4387 154  1024
 5     2.6774    0.9394  0.7312 154  1024
10     5.1139    1.5973  1.4624 154  1024
20     9.9712    2.9132  2.9248 154  1024